# XGBoost Model

The first HistGradientBoosting model reached a validation ROC-AUC of 0.940597.

For this experiment, I will test XGBoost on the same dataset and compare the result with the previous models.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv('../data/train.csv')

X = train.drop(columns=['Will_Buy_EV', 'id'])
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [ ]:
pipeline.fit(X_train, y_train)

valid_predictions = pipeline.predict_proba(X_valid)[:, 1]
xgb_auc = roc_auc_score(y_valid, valid_predictions)

print(f'XGBoost ROC-AUC: {xgb_auc:.6f}')

## Model Comparison

| Model | ROC-AUC |
|---|---:|
| Logistic Regression | 0.938000 |
| HistGradientBoosting | 0.940597 |
| XGBoost | See result above |